# 04 · Especialização: agente comparado ao RAG simples

Cobre o item **05** da pontuação. Das duas opções do enunciado (LoRA ou agente), a equipe fez o **agente**.

**Por que agente e não LoRA.** A máquina do projeto não tem GPU NVIDIA (o treino de LoRA em CPU não é viável no prazo), e os erros do RAG simples medidos na partição dev não eram de "falta de conhecimento do modelo": eram de **busca** (pergunta com dois assuntos, pergunta em linguagem muito coloquial) e de **falta de dado** (situação atual de um bairro, que não está em documento nenhum). Esses erros se resolvem com decisão e ferramentas, não com ajuste de pesos.

## O que o agente faz a mais

| Passo | RAG simples | Agente |
|---|---|---|
| Planejar | não tem | a LLM reescreve a pergunta em até 3 consultas (uma por assunto) e detecta se o morador quer a situação de um lugar |
| Ferramentas | 1 busca com a pergunta original | `buscar_base` (uma busca por consulta, mais a original) e `consultar_mapa` (export da camada Ouro do componente Dados) |
| Contexto | 3 trechos | até 5 trechos, os melhores de todas as buscas |
| Verificar | confere a evidência (só registra) | checagem determinística: todo número da resposta existe nos trechos? a resposta é feita das palavras dos trechos citados? |
| Repetir | não tem | se os trechos não respondem, replaneja com outras palavras; se a verificação reprova, regenera com a crítica; se não fechar em 2 rodadas, **recusa** |

Código: `src/boca_de_lobo_ia/agente.py` e `mapa.py`. Cada passo fica em `Resposta.passos`.

In [ ]:
import os, sys, json
from pathlib import Path
RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))
os.environ["MAPA_URL"] = ""   # notebooks usam a cópia local do mapa, para as saídas serem reproduzíveis
import pandas as pd
pd.set_option("display.max_colwidth", 200); pd.set_option("display.width", 220)
from boca_de_lobo_ia import agente, avaliacao, busca, config, geracao, lgpd
indice = busca.Indice()
LIMIAR = config.limiar_recusa()
print("modelo LLM:", config.MODELO_LLM, "| temperatura:", config.TEMPERATURA, "| semente:", config.SEMENTE, "| limiar:", LIMIAR)

def mostrar(r, campos=("configuracao", "recusa", "resposta", "evidencia", "evidencia_confere", "motivo", "confianca", "limiar", "chamadas_llm", "tempo_s")):
    d = r.para_dict()
    saida = {c: d[c] for c in campos}
    saida["fontes"] = [f.get("id") or f.get("descricao") for f in d["fontes"]]
    print(json.dumps(saida, ensure_ascii=False, indent=2))

def passos(r):
    for p in r.passos:
        print("  ", json.dumps(p, ensure_ascii=False)[:230])

## O prompt do planejador

In [ ]:
print(agente.SISTEMA_PLANEJAR)

## Demonstração 1 · pergunta com dois assuntos

O RAG simples faz uma busca só; os 3 trechos tendem a cobrir um dos assuntos. O agente quebra a pergunta em duas consultas.

In [ ]:
pergunta = "Entrou água em casa e perdi os móveis. Posso sacar o FGTS e também ter desconto no IPTU?"
rag = geracao.responder_rag(pergunta, indice)
mostrar(rag)
print("trechos no contexto do RAG:", [x["id"] for x in rag.recuperados])

In [ ]:
ag = agente.responder_agente(pergunta, indice)
mostrar(ag)
passos(ag)

## Demonstração 2 · pergunta que depende de dado do produto (ferramenta `consultar_mapa`)

"Tem bueiro entupido no Grajaú?" não está em documento nenhum: está no mapa que o componente Dados publica (`ouro_vw_mapa_cidadao`). O RAG simples só pode recusar ou falar de outra coisa. O agente consulta o mapa e cita o dado, com a data.

In [ ]:
pergunta = "Tem bueiro entupido no Grajaú hoje?"
rag = geracao.responder_rag(pergunta, indice)
mostrar(rag)

In [ ]:
ag = agente.responder_agente(pergunta, indice)
mostrar(ag)
passos(ag)

In [ ]:
# O "trecho" que a ferramenta devolveu ao modelo (dado, não documento):
from boca_de_lobo_ia import mapa
print(mapa.consultar("Grajaú").texto)

## Demonstração 3 · pergunta que não consta: o agente insiste uma vez e recusa

O benefício perguntado (auxílio-aluguel mensal) não está na base; há um parecido (Cartão Emergencial, pagamento único). O agente replaneja uma vez e, sem achar, recusa.

In [ ]:
pergunta = "Quanto a prefeitura paga por mês de auxílio-aluguel pra quem perdeu a casa na enchente?"
ag = agente.responder_agente(pergunta, indice)
mostrar(ag)
passos(ag)

## A verificação, isolada

As checagens do passo 4 são funções puras, testadas em `tests/test_nucleo.py`:

In [ ]:
trecho = "O valor do saque será limitado à quantia correspondente a R$ 6.220,00, desde que o intervalo entre uma movimentação e outra não seja inferior a doze meses."
for resposta in ["Você pode sacar até R$ 6.220,00.", "Você pode sacar até R$ 8.000 em até 30 dias."]:
    print(f"{resposta!r:55} números sem apoio: {agente.numeros_sem_apoio(resposta, [trecho], 'quanto posso sacar?')}"
          f" | apoio lexical: {geracao.apoio_lexical(resposta, [trecho]):.2f}")

## Comparação na partição dev (21 casos)

Respostas gravadas por `scripts/avaliar.py --particao dev`. O placar que vale é o da partição de teste (notebook 05); esta tabela mostra o que orientou os ajustes.

In [ ]:
colunas = ["configuracao", "casos", "acerto_pct", "alucinacao_pct", "recusa_indevida_pct", "recusa_no_nao_consta_pct", "citacao_correta_pct", "tempo_medio_s", "chamadas_llm_media"]
pd.DataFrame(avaliacao.placar("dev"))[colunas]

In [ ]:
r_rag, r_ag = avaliacao.carregar_resultados("rag"), avaliacao.carregar_resultados("agente")
casos = {c["id"]: c for c in avaliacao.carregar_casos("dev")}
pd.DataFrame([{
    "id": i, "categoria": casos[i]["categoria"], "pergunta": casos[i]["pergunta"][:70],
    "RAG": r_rag[i]["julgamento"]["veredito"], "agente": r_ag[i]["julgamento"]["veredito"],
    "tempo RAG (s)": r_rag[i]["tempo_s"], "tempo agente (s)": r_ag[i]["tempo_s"],
} for i in casos if i in r_rag and i in r_ag])

## Custo da especialização

O agente faz de 2 a 5 chamadas à LLM por pergunta (planejar, responder e, quando preciso, replanejar ou regenerar), contra 1 do RAG simples. Em CPU isso significa respostas várias vezes mais lentas (colunas de tempo acima). Para o morador em emergência, a tela mantém os telefones 199 e 193 visíveis enquanto a resposta é gerada.

Próximo: `05_placar_e_erros.ipynb`.